# Project: Identify Customer Segments

In this project, you will apply unsupervised learning techniques to identify segments of the population that form the core customer base for a mail-order sales company in Germany. These segments can then be used to direct marketing campaigns towards audiences that will have the highest expected rate of returns. The data that you will use has been provided by our partners at Bertelsmann Arvato Analytics, and represents a real-life data science task.

This notebook will help you complete this task by providing a framework within which you will perform your analysis steps. In each step of the project, you will see some text describing the subtask that you will perform, followed by one or more code cells for you to complete your work. **Feel free to add additional code and markdown cells as you go along so that you can explore everything in precise chunks.** The code cells provided in the base template will outline only the major tasks, and will usually not be enough to cover all of the minor tasks that comprise it.

It should be noted that while there will be precise guidelines on how you should handle certain tasks in the project, there will also be places where an exact specification is not provided. **There will be times in the project where you will need to make and justify your own decisions on how to treat the data.** These are places where there may not be only one way to handle the data. In real-life tasks, there may be many valid ways to approach an analysis task. One of the most important things you can do is clearly document your approach so that other scientists can understand the decisions you've made.

At the end of most sections, there will be a Markdown cell labeled **Discussion**. In these cells, you will report your findings for the completed section, as well as document the decisions that you made in your approach to each subtask. **Your project will be evaluated not just on the code used to complete the tasks outlined, but also your communication about your observations and conclusions at each stage.**

In [2]:
# import libraries here; add more as necessary
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from pprint import pp as pprint

# magic word for producing visualizations in notebook
%matplotlib inline

pd.options.display.max_rows = None
pd.options.display.max_columns = None

### Step 0: Load the Data

There are four files associated with this project (not including this one):

- `Udacity_AZDIAS_Subset.csv`: Demographics data for the general population of Germany; 891211 persons (rows) x 85 features (columns).
- `Udacity_CUSTOMERS_Subset.csv`: Demographics data for customers of a mail-order company; 191652 persons (rows) x 85 features (columns).
- `Data_Dictionary.md`: Detailed information file about the features in the provided datasets.
- `AZDIAS_Feature_Summary.csv`: Summary of feature attributes for demographics data; 85 features (rows) x 4 columns

Each row of the demographics files represents a single person, but also includes information outside of individuals, including information about their household, building, and neighborhood. You will use this information to cluster the general population into groups with similar demographic properties. Then, you will see how the people in the customers dataset fit into those created clusters. The hope here is that certain clusters are over-represented in the customers data, as compared to the general population; those over-represented clusters will be assumed to be part of the core userbase. This information can then be used for further applications, such as targeting for a marketing campaign.

To start off with, load in the demographics data for the general population into a pandas DataFrame, and do the same for the feature attributes summary. Note for all of the `.csv` data files in this project: they're semicolon (`;`) delimited, so you'll need an additional argument in your [`read_csv()`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.read_csv.html) call to read in the data properly. Also, considering the size of the main dataset, it may take some time for it to load completely.

Once the dataset is loaded, it's recommended that you take a little bit of time just browsing the general structure of the dataset and feature summary file. You'll be getting deep into the innards of the cleaning in the first major step of the project, so gaining some general familiarity can help you get your bearings.

In [3]:
# Load in the general demographics' data.
gen_data = pd.read_csv("Udacity_AZDIAS_Subset.csv", delimiter=";")

# Load in the feature summary file.
feature_summary = pd.read_csv("AZDIAS_Feature_Summary.csv", delimiter=";")

#### Exploring the General Population Data:

In [4]:
# Print the first 5 rows
gen_data.head()

,AGER_TYP,ALTERSKATEGORIE_GROB,ANREDE_KZ,CJT_GESAMTTYP,FINANZ_MINIMALIST,FINANZ_SPARER,FINANZ_VORSORGER,FINANZ_ANLEGER,FINANZ_UNAUFFAELLIGER,FINANZ_HAUSBAUER,FINANZTYP,GEBURTSJAHR,GFK_URLAUBERTYP,GREEN_AVANTGARDE,HEALTH_TYP,LP_LEBENSPHASE_FEIN,LP_LEBENSPHASE_GROB,LP_FAMILIE_FEIN,LP_FAMILIE_GROB,LP_STATUS_FEIN,LP_STATUS_GROB,NATIONALITAET_KZ,PRAEGENDE_JUGENDJAHRE,RETOURTYP_BK_S,SEMIO_SOZ,SEMIO_FAM,SEMIO_REL,SEMIO_MAT,SEMIO_VERT,SEMIO_LUST,SEMIO_ERL,SEMIO_KULT,SEMIO_RAT,SEMIO_KRIT,SEMIO_DOM,SEMIO_KAEM,SEMIO_PFLICHT,SEMIO_TRADV,SHOPPER_TYP,SOHO_KZ,TITEL_KZ,VERS_TYP,ZABEOTYP,ALTER_HH,ANZ_PERSONEN,ANZ_TITEL,HH_EINKOMMEN_SCORE,KK_KUNDENTYP,W_KEIT_KIND_HH,WOHNDAUER_2008,ANZ_HAUSHALTE_AKTIV,ANZ_HH_TITEL,GEBAEUDETYP,KONSUMNAEHE,MIN_GEBAEUDEJAHR,OST_WEST_KZ,WOHNLAGE,CAMEO_DEUG_2015,CAMEO_DEU_2015,CAMEO_INTL_2015,KBA05_ANTG1,KBA05_ANTG2,KBA05_ANTG3,KBA05_ANTG4,KBA05_BAUMAX,KBA05_GBZ,BALLRAUM,EWDICHTE,INNENSTADT,GEBAEUDETYP_RASTER,KKK,MOBI_REGIO,ONLINE_AFFINITAET,REGIOTYP,KBA13_ANZAHL_PKW,PLZ8_ANTG1,PLZ8_ANTG2,PLZ8_ANTG3,PLZ8_ANTG4,PLZ8_BAUMAX,PLZ8_HHZ,PLZ8_GBZ,ARBEIT,ORTSGR_KLS9,RELAT_AB
0,-1,2,1,2.0,3,4,3,5,5,3,4,0,10.0,0,-1,15.0,4.0,2.0,2.0,1.0,1.0,0,0,5.0,2,6,7,5,1,5,3,3,4,7,6,6,5,3,-1,NaN,NaN,-1,3,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-1,1,2,5.0,1,5,2,5,4,5,1,1996,10.0,0,3,21.0,6.0,5.0,3.0,2.0,1.0,1,14,1.0,5,4,4,3,1,2,2,3,6,4,7,4,7,6,3,1.0,0.0,2,5,0.0,2.0,0.0,6.0,NaN,3.0,9.0,11.0,0.0,8.0,1.0,1992.0,W,4.0,8,8A,51,0.0,0.0,0.0,2.0,5.0,1.0,6.0,3.0,8.0,3.0,2.0,1.0,3.0,3.0,963.0,2.0,3.0,2.0,1.0,1.0,5.0,4.0,3.0,5.0,4.0
2,-1,3,2,3.0,1,4,1,2,3,5,1,1979,10.0,1,3,3.0,1.0,1.0,1.0,3.0,2.0,1,15,3.0,4,1,3,3,4,4,6,3,4,7,7,7,3,3,2,0.0,0.0,1,5,17.0,1.0,0.0,4.0,NaN,3.0,9.0,10.0,0.0,1.0,5.0,1992.0,W,2.0,4,4C,24,1.0,3.0,1.0,0.0,0.0,3.0,2.0,4.0,4.0,4.0,2.0,3.0,2.0,2.0,712.0,3.0,3.0,1.0,0.0,1.0,4.0,4.0,3.0,5.0,2.0
3,2,4,2,2.0,4,2,5,2,1,2,6,1957,1.0,0,2,0.0,0.0,0.0,0.0,9.0,4.0,1,8,2.0,5,1,2,1,4,4,7,4,3,4,4,5,4,4,1,0.0,0.0,1,3,13.0,0.0,0.0,1.0,NaN,NaN,9.0,1.0,0.0,1.0,4.0,1997.0,W,7.0,2,2A,12,4.0,1.0,0.0,0.0,1.0,4.0,4.0,2.0,6.0,4.0,0.0,4.0,1.0,0.0,596.0,2.0,2.0,2.0,0.0,1.0,3.0,4.0,2.0,3.0,3.0
4,-1,3,1,5.0,4,3,4,1,3,2,5,1963,5.0,0,3,32.0,10.0,10.0,5.0,3.0,2.0,1,8,5.0,6,4,4,2,7,4,4,6,2,3,2,2,4,2,2,0.0,0.0,2,4,20.0,4.0,0.0,5.0,1.0,2.0,9.0,3.0,0.0,1.0,4.0,1992.0,W,3.0,6,6B,43,1.0,4.0,1.0,0.0,0.0,3.0,2.0,5.0,1.0,5.0,3.0,3.0,5.0,5.0,435.0,2.0,4.0,2.0,1.0,2.0,3.0,3.0,4.0,6.0,5.0


In [5]:
# Get some more information about the data
gen_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891221 entries, 0 to 891220
Data columns (total 85 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   AGER_TYP               891221 non-null  int64  
 1   ALTERSKATEGORIE_GROB   891221 non-null  int64  
 2   ANREDE_KZ              891221 non-null  int64  
 3   CJT_GESAMTTYP          886367 non-null  float64
 4   FINANZ_MINIMALIST      891221 non-null  int64  
 5   FINANZ_SPARER          891221 non-null  int64  
 6   FINANZ_VORSORGER       891221 non-null  int64  
 7   FINANZ_ANLEGER         891221 non-null  int64  
 8   FINANZ_UNAUFFAELLIGER  891221 non-null  int64  
 9   FINANZ_HAUSBAUER       891221 non-null  int64  
 10  FINANZTYP              891221 non-null  int64  
 11  GEBURTSJAHR            891221 non-null  int64  
 12  GFK_URLAUBERTYP        886367 non-null  float64
 13  GREEN_AVANTGARDE       891221 non-null  int64  
 14  HEALTH_TYP             891221 non-nu

In [6]:
# Use .describe() to get summary statistics
gen_data.describe()

,AGER_TYP,ALTERSKATEGORIE_GROB,ANREDE_KZ,CJT_GESAMTTYP,FINANZ_MINIMALIST,FINANZ_SPARER,FINANZ_VORSORGER,FINANZ_ANLEGER,FINANZ_UNAUFFAELLIGER,FINANZ_HAUSBAUER,FINANZTYP,GEBURTSJAHR,GFK_URLAUBERTYP,GREEN_AVANTGARDE,HEALTH_TYP,LP_LEBENSPHASE_FEIN,LP_LEBENSPHASE_GROB,LP_FAMILIE_FEIN,LP_FAMILIE_GROB,LP_STATUS_FEIN,LP_STATUS_GROB,NATIONALITAET_KZ,PRAEGENDE_JUGENDJAHRE,RETOURTYP_BK_S,SEMIO_SOZ,SEMIO_FAM,SEMIO_REL,SEMIO_MAT,SEMIO_VERT,SEMIO_LUST,SEMIO_ERL,SEMIO_KULT,SEMIO_RAT,SEMIO_KRIT,SEMIO_DOM,SEMIO_KAEM,SEMIO_PFLICHT,SEMIO_TRADV,SHOPPER_TYP,SOHO_KZ,TITEL_KZ,VERS_TYP,ZABEOTYP,ALTER_HH,ANZ_PERSONEN,ANZ_TITEL,HH_EINKOMMEN_SCORE,KK_KUNDENTYP,W_KEIT_KIND_HH,WOHNDAUER_2008,ANZ_HAUSHALTE_AKTIV,ANZ_HH_TITEL,GEBAEUDETYP,KONSUMNAEHE,MIN_GEBAEUDEJAHR,WOHNLAGE,KBA05_ANTG1,KBA05_ANTG2,KBA05_ANTG3,KBA05_ANTG4,KBA05_BAUMAX,KBA05_GBZ,BALLRAUM,EWDICHTE,INNENSTADT,GEBAEUDETYP_RASTER,KKK,MOBI_REGIO,ONLINE_AFFINITAET,REGIOTYP,KBA13_ANZAHL_PKW,PLZ8_ANTG1,PLZ8_ANTG2,PLZ8_ANTG3,PLZ8_ANTG4,PLZ8_BAUMAX,PLZ8_HHZ,PLZ8_GBZ,ARBEIT,ORTSGR_KLS9,RELAT_AB
count,891221.000000,891221.000000,891221.000000,886367.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,886367.000000,891221.000000,891221.000000,886367.000000,886367.000000,886367.000000,886367.000000,886367.000000,886367.000000,891221.000000,891221.000000,886367.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,891221.000000,817722.000000,817722.000000,891221.000000,891221.000000,817722.000000,817722.000000,817722.000000,872873.000000,306609.000000,783619.000000,817722.000000,798073.000000,794213.000000,798073.000000,817252.000000,798073.000000,798073.000000,757897.000000,757897.000000,757897.000000,757897.000000,757897.000000,757897.000000,797481.000000,797481.000000,797481.000000,798066.000000,770025.000000,757897.000000,886367.000000,770025.000000,785421.000000,774706.000000,774706.000000,774706.000000,774706.000000,774706.000000,774706.000000,774706.000000,794005.000000,794005.000000,794005.00000
mean,-0.358435,2.777398,1.522098,3.632838,3.074528,2.821039,3.401106,3.033328,2.874167,3.075121,3.790586,1101.178533,7.350304,0.196612,1.792102,14.622637,4.453621,3.599574,2.185966,4.791151,2.432575,1.026827,8.154346,3.419630,3.945860,4.272729,4.240609,4.001597,4.023709,4.359086,4.481405,4.025014,3.910139,4.763223,4.667550,4.445007,4.256076,3.661784,1.266967,0.008423,0.003483,1.197852,3.362438,10.864126,1.727637,0.004162,4.207243,3.410640,3.933406,7.908791,8.287263,0.040647,2.798641,3.018452,1993.277011,4.052836,1.494277,1.265584,0.624525,0.305927,1.389552,3.158580,4.153043,3.939172,4.549491,3.738306,2.592991,2.963540,2.698691,4.257967,619.701439,2.253330,2.801858,1.595426,0.699166,1.943913,3.612821,3.381087,3.167854,5.293002,3.07222
std,1.198724,1.068775,0.499512,1.595021,1.321055,1.464749,1.322134,1.529603,1.486731,1.353248,1.987876,976.583551,3.525723,0.397437,1.269062,12.616883,3.855639,3.926486,1.756537,3.425305,1.474315,0.586634,4.844532,1.417741,1.946564,1.915885,2.007373,1.857540,2.077746,2.022829,1.807552,1.903816,1.580306,1.830789,1.795712,1.852412,1.770137,1.707637,1.287435,0.091392,0.084957,0.952532,1.352704,7.639683,1.155849,0.068855,1.624057,1.628844,1.964701,1.923137,15.628087,0.324028,2.656713,1.550312,3.332739,1.949539,1.403961,1.245178,1.013443,0.638725,1.779483,1.329537,2.183710,1.718996,2.028919,0.923193,1.119052,1.428882,1.521524,2.030385,340.034318,0.972008,0.920309,0.986736,0.727137,1.459654,0.973967,1.111598,1.002376,2.303739,1.36298
min,-1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000

In [7]:
# Get a peek of the data types
gen_data.dtypes.value_counts()

float64    49
int64      32
object      4
Name: count, dtype: int64

So basic information about the general population demographics data:
- There are 891221 entries/samples, or rows, each representing one person
- There are a total of 85 rows or features
- There are 81 numeric features (32xint64 + 49xfloat64) and 4 non-numeric features

Now we explore how many NaN values are included in the data:

In [8]:
# Calculate the total number of NaN values in the dataset
print(f"Number of NaN values in demographics dataset: {gen_data.isna().sum().sum()}")

# Get the count of NaN values per column or feature
gen_data.isna().sum().sort_values(ascending=False)

Number of NaN values in demographics dataset: 4896838


KK_KUNDENTYP             584612
KBA05_GBZ                133324
KBA05_ANTG1              133324
KBA05_ANTG3              133324
MOBI_REGIO               133324
KBA05_ANTG2              133324
KBA05_ANTG4              133324
KBA05_BAUMAX             133324
REGIOTYP                 121196
KKK                      121196
PLZ8_HHZ                 116515
PLZ8_BAUMAX              116515
PLZ8_ANTG4               116515
PLZ8_GBZ                 116515
PLZ8_ANTG2               116515
PLZ8_ANTG1               116515
PLZ8_ANTG3               116515
W_KEIT_KIND_HH           107602
KBA13_ANZAHL_PKW         105800
CAMEO_INTL_2015           98979
CAMEO_DEU_2015            98979
CAMEO_DEUG_2015           98979
ARBEIT                    97216
ORTSGR_KLS9               97216
RELAT_AB                  97216
ANZ_HH_TITEL              97008
BALLRAUM                  93740
INNENSTADT                93740
EWDICHTE                  93740
GEBAEUDETYP_RASTER        93155
WOHNLAGE                  93148
ANZ_HAUS

As we can see, several features have NaN values, totaling 4,896,838. **NOTE**: this are actual NaN values recorded in the data. There could still be missings or unknowns in the data.

#### Exploring Feature Summary Data:

In [9]:
feature_summary.head()

,attribute,information_level,type,missing_or_unknown
0,AGER_TYP,person,categorical,"[-1,0]"
1,ALTERSKATEGORIE_GROB,person,ordinal,"[-1,0,9]"
2,ANREDE_KZ,person,categorical,"[-1,0]"
3,CJT_GESAMTTYP,person,categorical,[0]
4,FINANZ_MINIMALIST,person,ordinal,[-1]


In [10]:
feature_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   attribute           85 non-null     object
 1   information_level   85 non-null     object
 2   type                85 non-null     object
 3   missing_or_unknown  85 non-null     object
dtypes: object(4)
memory usage: 2.8+ KB


In [11]:
# Explore the "information_level" column. Get the values and counts
feature_summary["information_level"].value_counts()

information_level
person            43
macrocell_plz8     8
household          7
building           7
microcell_rr3      6
region_rr1         5
microcell_rr4      3
postcode           3
community          3
Name: count, dtype: int64

In [12]:
# Explore the "type" column. Get the values and counts
feature_summary["type"].value_counts()

type
ordinal        49
categorical    21
numeric         7
mixed           7
interval        1
Name: count, dtype: int64

The feature summary basically gives us the following information about each attribute/column/feature:
- attribute: the name of the attribute/feature
- information_level: attributes can be personal or higher-level (ex. geographical)
- type: the kind of data that an attribute or feature contains: orinal (ranked), categorical (labels/codes), numeric (numbers), mixed or interval
- missing_or_unknown: values for missing or unknown attribute data

#### Loading Data Dictionary:

I decided to load the Data Dictionary from a JSON file so that I can reference it in this notebook.

In [21]:
import json
# Load Data Dictionary
with open("data_dictionary.json") as f:
    data_dictionary_list = json.load(f)

data_dictionary = {}

for d in data_dictionary_list:
    data_dictionary[d["feature_name"]] = {
        "desc": d["desc"],
        "type": feature_summary[feature_summary["attribute"] == d["feature_name"]]["type"].item(),
        "codes": d["codes"]
    }

pprint(data_dictionary)

{'AGER_TYP': {'desc': 'Best-ager typology',
              'type': 'categorical',
              'codes': {'-1': 'unknown',
                        '0': 'no classification possible',
                        '1': 'passive elderly',
                        '2': 'cultural elderly',
                        '3': 'experience-driven elderly'}},
 'ALTERSKATEGORIE_GROB': {'desc': 'Estimated age based on given name analysis '
                                  '(rough categories)',
                          'type': 'ordinal',
                          'codes': {'-1': 'unknown (missing)',
                                    '0': 'unknown (cannot be determined)',
                                    '1': '< 30 years old',
                                    '2': '30 - 45 years old',
                                    '3': '46 - 60 years old',
                                    '4': '> 60 years old',
                                    '9': 'uniformly distributed'}},
 'ANREDE_KZ': {'desc': 'Gender (s

> **Tip**: Add additional cells to keep everything in reasonably-sized chunks! Keyboard shortcut `esc --> a` (press escape to enter command mode, then press the 'A' key) adds a new cell before the active cell, and `esc --> b` adds a new cell after the active cell. If you need to convert an active cell to a markdown cell, use `esc --> m` and to convert to a code cell, use `esc --> y`. 

## Step 1: Preprocessing

### Step 1.1: Assess Missing Data

The feature summary file contains a summary of properties for each demographics data column. You will use this file to help you make cleaning decisions during this stage of the project. First of all, you should assess the demographics data in terms of missing data. Pay attention to the following points as you perform your analysis, and take notes on what you observe. Make sure that you fill in the **Discussion** cell with your findings and decisions at the end of each step that has one!

#### Step 1.1.1: Convert Missing Value Codes to NaNs
The fourth column of the feature attributes summary (loaded in above as `feat_info`) documents the codes from the data dictionary that indicate missing or unknown data. While the file encodes this as a list (e.g. `[-1,0]`), this will get read in as a string object. You'll need to do a little bit of parsing to make use of it to identify and clean the data. Convert data that matches a 'missing' or 'unknown' value code into a numpy NaN value. You might want to see how much data takes on a 'missing' or 'unknown' code, and how much data is naturally missing, as a point of interest.

**As one more reminder, you are encouraged to add additional cells to break up your analysis into manageable chunks.**

In [ ]:
# Real NaN values
real_nan_total = gen_data.isnull().sum().sum()
print(f"Total Real Missing Data: {real_nan_total}")

#### Identifying Missing Data:

In [ ]:
# Identify missing or unknown data values and convert them to NaNs.
for attribute, missing_encoded_str in zip(feature_summary["attribute"], feature_summary["missing_or_unknown"]):
    print(f"Feature {attribute} encodes missing values as: {missing_encoded_str}, {type(missing_encoded_str)}")

Convert missing_or_unknown strings into actual lists of values:

In [ ]:
def get_numeric_value(s):
    """
    Basic function that attempts to cast a string into an int.
    :param s: the string to cast to int
    :return: the integer represented by the string or the original string if invalid string
    """
    try:
        return int(s)
    except ValueError:
        return s

# Init dictionary
miss_or_unkwn = {}

# iterate over attributes (to use as key) and missing_or_unknown values (as strings now)
for attribute, missing_encoded_str in zip(feature_summary["attribute"], feature_summary["missing_or_unknown"]):

    # clean up the string and create a list
    temp_list = list(missing_encoded_str.replace(" ", "").replace("[","").replace("]","").split(","))

    # check that the list is not empty
    if len(temp_list) > 0 and temp_list[0] != "":
        # convert strings into numerical values
        miss_or_unkwn[attribute] = [get_numeric_value(s) for s in temp_list]

print(f"miss_or_unkwn dictionary: {miss_or_unkwn}")

Now we go over all columns having missing or unknown values and replace them with NaN:

In [ ]:
# keep track of total missing or unknown datums (so we can have a separate value for real NaN values in the data
# and converted NaN values). Used later for sanity check.
converted_nan_total = 0

# we iterate over each attribute and list pair in the dictionary
# we don't iterate over each column, as not all of them have missing_or_unknown codes
for attribute, lst in miss_or_unkwn.items():

    # Get the sum of missing or unknown values for that attribute/feature in the general pop. data
    # We use isin() function to match with the list of missing_or_unkown values and then sum them up
    n_miss_unkwn = gen_data[attribute].isin(lst).sum()

    # Get the count of real NaN values in the dataset for that feature
    n_nan = gen_data[attribute].isna().sum()

    print(f"{attribute} has {n_miss_unkwn} missing or unknowns and {n_nan} real NaN values.")

    # Replace all the values with NaN
    gen_data[attribute] = gen_data[attribute].replace(lst, np.nan)

    # increment total
    converted_nan_total += n_miss_unkwn

print(f"Total number of missing values: {converted_nan_total}")

**Sanity Check**: checking that we correctly converted all NaN values in columns

In [ ]:
# Sum of naturally missing + missing_or_unknown values
nan_total = converted_nan_total + real_nan_total
print(nan_total)

In [ ]:
# If all is good, then the whole dataset should have the same total number as above
print(f"Number of null values in general population dataset: {gen_data.isna().sum().sum()}")
assert nan_total == gen_data.isna().sum().sum()

# all good, output a summary of the NaN values per feature
gen_data.isna().sum().sort_values(ascending=False)

#### Step 1.1.2: Assess Missing Data in Each Column

How much missing data is present in each column? There are a few columns that are outliers in terms of the proportion of values that are missing. You will want to use matplotlib's [`hist()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.hist.html) function to visualize the distribution of missing value counts to find these columns. Identify and document these columns. While some of these columns might have justifications for keeping or re-encoding the data, for this project you should just remove them from the dataframe. (Feel free to make remarks about these outlier columns in the discussion, however!)

For the remaining features, are there any patterns in which columns have, or share, missing data?

In [ ]:
# Perform an assessment of how much missing data there is in each column of the
# dataset.
count_miss_values = gen_data.isna().sum()
print(count_miss_values)

In [ ]:
# Plot Counts
plt.figure(figsize=(10,6))
plt.bar(count_miss_values.index, count_miss_values.values, color="b", edgecolor='black')
plt.title('Missing Values per Feature/Column')
plt.xlabel('Feature/Column')
plt.xticks(rotation=90, ha="center", fontsize=6)
plt.ylabel('Number of NaN values')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Convert to percentages
pct_missing_values = (count_miss_values / len(gen_data)) * 100
pct_missing_values.sort_values(ascending=False)

In [ ]:
# Plot Percentages
plt.figure(figsize=(10,6))
plt.bar(pct_missing_values.index, pct_missing_values.values, color="r", edgecolor='black')
plt.title('Percentage of Missing Values per Feature/Column')
plt.xlabel('Feature/Column')
plt.xticks(rotation=90, ha="center", fontsize=6)
plt.ylabel('% of Missing (NaN) Values')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

As can be seen in the bar graph, a possible good threshold would be 20% of missing values

In [ ]:
# Use pd.cut() to separate each column's percentage of missing values into defined ranges (bins)
bins = [0, 10, 20, 30, 40, 50,75, 100]
labels = ["0–10%", "10–20%", "20-30%", "30-40%", "40–50%",'50-75%', '75–100%']
grouped = pd.cut(pct_missing_values, bins=bins, labels=labels, include_lowest=True)

# Summarize by getting the count of each value and sorting
print(grouped.value_counts().sort_index())

As can be seen, out of the 85 features/columns, 79 columns have 20% or less missing values, and 6 columns have more than 20%.

In [ ]:
# Remove the outlier columns from the dataset. (You'll perform other data
# engineering tasks such as re-encoding and imputation later.)

# Get a list of the columns that have > 20% missing values
drop_cols = list(pct_missing_values[pct_missing_values > 20].index)
print(f"Columns to Drop: {drop_cols}")

In [ ]:
# Feature Summary for the columns that we will drop
drop_summary = feature_summary[feature_summary["attribute"].isin(drop_cols)]
drop_summary

In [ ]:
# Drop the columns
gen_data.drop(drop_cols, axis=1, inplace=True)

# Sanity check, there should be 79 remaining columns
assert len(gen_data.columns) == 79

#### Discussion 1.1.2: Assess Missing Data in Each Column

As can be seen above from the graph and the summary of percentage missing values, the number of missing values is not uniformly distributed across the dataset. 39/85 columns are missing 0-10% and 40/85 columns are missing 10-20% of values. This can be remedied later with imputation. I decided to use 20% missing values as a threshold based on the information gathered.

The columns dropped are:
- **'AGER_TYP'**: Best-ager typology. >70% missing data. Not useful.
- **'GEBURTSJAHR'**: Year of birth. >40% missing data. Not useful.
- **'TITEL_KZ'**: Academic title flag. >99% missing data.
- **'ALTER_HH'**: Birthdate of head of household. >30% missing data. Not useful.
- **'KK_KUNDENTYP'**: Consumer pattern over past 12 months. >65% missing data. Not useful.
- **'KBA05_BAUMAX'**: Most common building type within microcell. >50% missing data. Not useful and probably irrelevant.

#### Step 1.1.3: Assess Missing Data in Each Row

Now, you'll perform a similar assessment for the rows of the dataset. How much data is missing in each row? As with the columns, you should see some groups of points that have a very different numbers of missing values. Divide the data into two subsets: one for data points that are above some threshold for missing values, and a second subset for points below that threshold.

In order to know what to do with the outlier rows, we should see if the distribution of data values on columns that are not missing data (or are missing very little data) are similar or different between the two groups. Select at least five of these columns and compare the distribution of values.
- You can use seaborn's [`countplot()`](https://seaborn.pydata.org/generated/seaborn.countplot.html) function to create a bar chart of code frequencies and matplotlib's [`subplot()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.subplot.html) function to put bar charts for the two subplots side by side.
- To reduce repeated code, you might want to write a function that can perform this comparison, taking as one of its arguments a column to be compared.

Depending on what you observe in your comparison, this will have implications on how you approach your conclusions later in the analysis. If the distributions of non-missing features look similar between the data with many missing values and the data with few or no missing values, then we could argue that simply dropping those points from the analysis won't present a major issue. On the other hand, if the data with many missing values looks very different from the data with few or no missing values, then we should make a note on those data as special. We'll revisit these data later on. **Either way, you should continue your analysis for now using just the subset of the data with few or no missing values.**

In [ ]:
# How much data is missing in each row of the dataset?

# Get the sum of NaN values per row
missing_per_row = gen_data.isna().sum(axis=1)
print(missing_per_row.describe())

In [ ]:
# For convenience, we turn into a Data Frame
missing_per_row_df = pd.DataFrame(missing_per_row, columns=["# of missing values"])
missing_per_row_df.head(20)

In [ ]:
# Plot a histogram
plt.hist(missing_per_row, bins=50);
plt.xlabel('Number of Missing Values')
plt.ylabel('Number of Rows')
plt.title('Distribution of Missing Values per Row', fontweight='bold')
plt.show()

As can be seen by the histogram, most of the rows are complete, with a small spike of rows missing

In [ ]:
# Plot again, focusing on rows that have <= 20 missing values
plt.hist(missing_per_row[missing_per_row <= 20], bins=20)
plt.xlabel('Number of Missing Values')
plt.ylabel('Number of Rows')
plt.title('Distribution of Missing Values per Row for Rows Missing 20 or less Values', fontweight='bold')
plt.show()

In [ ]:
# Write code to divide the data into two subsets based on the number of missing
# values in each row.

# Separate into two subsets, accept and reject, based on the threshold of 20 missing values per row
gen_subset_reject = gen_data[gen_data.isna().sum(axis=1) > 20]
gen_subset_accept = gen_data[gen_data.isna().sum(axis=1) <= 20]

In [ ]:
print(f"Total # of rows: {len(gen_data)}")
print(f"Total # of missing values in rows: {missing_per_row_df.sum().item()}")
print(f"Total # of rows in accept subset: {len(gen_subset_accept)}")
print(f"Total # of missing values in rows of accept subset: {gen_subset_accept.isna().sum().sum()}")

We have therefore decreased the number of rows from 891,221 to 797,426 (\~10% decrease) and the total number of missing values from 5,035,304 to 999,462 (\~80% decrease).

#### Comparing Distribution of Values Between Accepted and Rejected Subsets

In [ ]:
import random
random.seed(42)

# Select 5 features/columns that have 0 missing values
# First we get all columns with 0 missing values
candidate_cols = pct_missing_values[pct_missing_values == 0].index.to_list()

# Then we choose 5 random ones
comp_cols = random.choices(candidate_cols, k=5)
print(f"Columns/Features that will be compared: {comp_cols}")

In [ ]:
# Compare the distribution of values for at least five columns where there are
# no or few missing values, between the two subsets.
def plot_subset_missing_dist(col_label):
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_figwidth(15)

    sns.countplot(x=col_label, data=gen_subset_accept, ax=ax1)
    sns.countplot(x=col_label, data=gen_subset_reject, ax=ax2)

    plt.show()

In [ ]:
for c in comp_cols:
    plot_subset_missing_dist(c)

#### Discussion 1.1.3: Assess Missing Data in Each Row

As seen in the cells above, for missing_per_row, the mean is 5.6, so ~6 features are missing per row, which I would say is rather acceptable. 50% of the data has complete rows, and 75% is missing only 3 values. STD is 13.23, which is rather high. What we observe then is that there are rows that have many missing values (up to 49 max), which are simply unreliable and should be dropped (or not used, as per our subset data).

Comparison of the plots shows that the distributions between accepted and rejected are different, so we cannot simply drop those rows. However, despite the fact that we cannot drop the rows, we can notice on the rejected subset that some features have biased distributions since we can see bars on the chart that are dominated by a single value/category.

### Step 1.2: Select and Re-Encode Features

Checking for missing data isn't the only way in which you can prepare a dataset for analysis. Since the unsupervised learning techniques to be used will only work on data that is encoded numerically, you need to make a few encoding changes or additional assumptions to be able to make progress. In addition, while almost all of the values in the dataset are encoded using numbers, not all of them represent numeric values. Check the third column of the feature summary (`feat_info`) for a summary of types of measurement.
- For numeric and interval data, these features can be kept without changes.
- Most of the variables in the dataset are ordinal in nature. While ordinal values may technically be non-linear in spacing, make the simplifying assumption that the ordinal variables can be treated as being interval in nature (that is, kept without any changes).
- Special handling may be necessary for the remaining two variable types: categorical, and 'mixed'.

In the first two parts of this sub-step, you will perform an investigation of the categorical and mixed-type features and make a decision on each of them, whether you will keep, drop, or re-encode each. Then, in the last part, you will create a new data frame with only the selected and engineered columns.

Data wrangling is often the trickiest part of the data analysis process, and there's a lot of it to be done here. But stick with it: once you're done with this step, you'll be ready to get to the machine learning parts of the project!

In [ ]:
# How many features are there of each data type?

# get a list of columns in our gen_subset_accept data
data_cols = gen_subset_accept.columns.to_list()

# Index only those attributes that are in our gen_accept_subset
feature_summary[feature_summary["attribute"].isin(data_cols)]["type"].value_counts()

#### Step 1.2.1: Re-Encode Categorical Features

For categorical data, you would ordinarily need to encode the levels as dummy variables. Depending on the number of categories, perform one of the following:
- For binary (two-level) categoricals that take numeric values, you can keep them without needing to do anything.
- There is one binary variable that takes on non-numeric values. For this one, you need to re-encode the values as numbers or create a dummy variable.
- For multi-level categoricals (three or more values), you can choose to encode the values using multiple dummy variables (e.g. via [OneHotEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)), or (to keep things straightforward) just drop them from the analysis. As always, document your choices in the Discussion section.

In [ ]:
# Assess categorical variables: which are binary, which are multi-level, and
# which one needs to be re-encoded?

# Get a list of all the categorical features
cat_features = list(feature_summary[
                        (feature_summary["attribute"].isin(data_cols)) &
                        (feature_summary["type"] == "categorical")]["attribute"])

print(cat_features)

#### Summary of multi-level categoricals:

- **ANREDE_KZ**: Gender. Could be useful. We keep.
- **CJT_GESAMTTYP**: Customer-Journey-Typology: preferred information and buying channels for consumer. Relevant, keep.
- **FINANZTYP**: Most descriptive financial type for individual. Relevant, keep.
- **GFK_URLAUBERTYP**: Vacation habits. Could be relevant, keep.
- **GREEN_AVANTGARDE**: Membership in environmental sustainability as part of youth. Not really useful for marketing segmentation, drop.
- **LP_FAMILIE_FEIN**: Family type, fine scale. Relevant, keep.
- **LP_FAMILIE_GROB**: Family type, rough scale. Drop, as we keep the fine version
- **LP_STATUS_FEIN**: Social status, fine scale. Relevant, keep.
- **LP_STATUS_GROB**: Social status, rough scale Drop, as we keep the fine version.
- **NATIONALITAET_KZ**: Nationality based on given name analysis. Very broad and few categories. Will drop.
- **SHOPPER_TYP**: Shopper typology. Relevant, keep.
- **SOHO_KZ**: Small office / home office flag. Don't think it is too relevant. I think the other features can capture better information. Drop.
- **VERS_TYP**: Insurance typology. Could be relevant, keep.
- **ZABEOTYP**: Energy consumption typology. Could be relevant, keep.
- **KK_KUNDENTYP**: Consumer pattern over past 12 months. Definitely keep.
- **GEBAEUDETYP**: Type of building (residential vs. commercial). Will drop because I don't think it will capture useful information.
- **OST_WEST_KZ**: Building location via former East / West Germany. I don't know if former East/West Germany can influence consumer behavior. Will keep.
- **CAMEO_DEUG_2015**: German CAMEO: Wealth / Life Stage Typology, rough scale. Relevant, keep.
- **CAMEO_DEU_2015**: German CAMEO: Wealth / Life Stage Typology, detailed scale. Doesn't map to CAMEO_DEUG_2015. Relevant, keep.

In [ ]:
# Define the features/columns to drop as per the summary above
cat_features_drop = ["GREEN_AVANTGARDE", "LP_FAMILIE_GROB", "LP_STATUS_GROB", "NATIONALITAET_KZ", "SHOPPER_TYP", "SOHO_KZ", "GEBAEUDETYP"]

In [ ]:
# init lists to store features based on the type
binary_cats = [] # For binary categorical features
multi_cats = [] # For multi-level categorical

# Iterate over each categorical feature
for cat in cat_features:

    # Make sure the feature column still exists in the data we will use
    if cat in gen_subset_accept.columns:

        # Print some helpful information
        print(f"{cat}, unique values: {gen_subset_accept[cat].unique()}, # unique: {gen_subset_accept[cat].nunique()}")

        # Check if binary (two-level) categorical
        if gen_subset_accept[cat].nunique() == 2:
            # append to list
            binary_cats.append(cat)
        # check for multi-level categorical, the ones we will not drop
        elif cat not in cat_features_drop:
            multi_cats.append(cat)

print()
print(f"binary_cats before: {binary_cats}")
print()

# Print summary of categorical features
print("================================================================")
print("=============== SUMMARY OF CATEGORICAL FEATURES: ===============")
print()
print(f"multi_cats (keep): {multi_cats}")
print()
print(f"cat_features_drop (drop): {cat_features_drop}")
print()

In [ ]:
gen_subset_accept["OST_WEST_KZ"].unique()

In [ ]:
# Re-encode binary categorical feature(s) to be kept in the analysis
# BEFORE
print(f"Unique values before encoding: {gen_subset_accept["OST_WEST_KZ"].unique()}")

gen_subset_accept = gen_subset_accept.copy()

# Silence FutureWarning (very annoying)
with pd.option_context('future.no_silent_downcasting', True):
    # Then do the replacement with explicit type conversion to avoid FutureWarning
    gen_subset_accept["OST_WEST_KZ"] = gen_subset_accept["OST_WEST_KZ"].replace(
        {
            'O': 0,
            'W': 1
        }
    ).astype('int64')

# AFTER
print(f"Unique values after encoding: {gen_subset_accept["OST_WEST_KZ"].unique()}")

# CHECK TYPE
print(gen_subset_accept["OST_WEST_KZ"].dtypes)

In [ ]:
# One Hot Encode
gen_accept_encoded = pd.get_dummies(gen_subset_accept, columns=multi_cats)
gen_accept_encoded.drop(columns=cat_features_drop, inplace=True)
gen_accept_encoded.info()

#### Discussion 1.2.1: Re-Encode Categorical Features

(Double-click this cell and replace this text with your own text, reporting your findings and decisions regarding categorical features. Which ones did you keep, which did you drop, and what engineering steps did you perform?)

#### Step 1.2.2: Engineer Mixed-Type Features

There are a handful of features that are marked as "mixed" in the feature summary that require special treatment in order to be included in the analysis. There are two in particular that deserve attention; the handling of the rest are up to your own choices:
- "PRAEGENDE_JUGENDJAHRE" combines information on three dimensions: generation by decade, movement (mainstream vs. avantgarde), and nation (east vs. west). While there aren't enough levels to disentangle east from west, you should create two new variables to capture the other two dimensions: an interval-type variable for decade, and a binary variable for movement.
- "CAMEO_INTL_2015" combines information on two axes: wealth and life stage. Break up the two-digit codes by their 'tens'-place and 'ones'-place digits into two new ordinal variables (which, for the purposes of this project, is equivalent to just treating them as their raw numeric values).
- If you decide to keep or engineer new features around the other mixed-type features, make sure you note your steps in the Discussion section.

Be sure to check `Data_Dictionary.md` for the details needed to finish these tasks.

In [ ]:
# Investigate "PRAEGENDE_JUGENDJAHRE" and engineer two new variables.
gen_accept_encoded.loc[:, "PRAEGENDE_JUGENDJAHRE"].unique()

In [ ]:
# Create mapping dictionaries, using the Data_Dictionary.md as reference

# Map to decades 40s to 90s
map_to_decade = {
    1: 4,
    2: 4,
    3: 5,
    4: 5,
    5: 6,
    6: 6,
    7: 6,
    8: 7,
    9: 7,
    10: 8,
    11: 8,
    12: 8,
    13: 8,
    14: 9,
    15: 9
}

# Map to movement
# 1 = Mainstream, 0 = Avantgrade
map_to_movement = {
    1: 1,
    2: 0,
    3: 1,
    4: 0,
    5: 1,
    6: 0,
    7: 0,
    8: 1,
    9: 0,
    10: 1,
    11: 0,
    12: 1,
    13: 0,
    14: 1,
    15: 0
}

In [ ]:
gen_accept_encoded["DECADE"] = gen_subset_accept.loc[:, "PRAEGENDE_JUGENDJAHRE"].map(map_to_decade)
gen_accept_encoded["MOVEMENT"] = gen_subset_accept.loc[:, "PRAEGENDE_JUGENDJAHRE"].map(map_to_movement)

In [ ]:
# Sanity check
gen_accept_encoded.loc[:, ["PRAEGENDE_JUGENDJAHRE", "DECADE", "MOVEMENT"]].head(10)

In [ ]:
# Investigate "CAMEO_INTL_2015" and engineer two new variables.
# German CAMEO: Wealth / Life Stage Typology, mapped to international code
gen_accept_encoded.loc[:, "CAMEO_INTL_2015"].unique()

In [ ]:
# Since we are separating the digits, rather than mapping, I will attempt to perform
# an operation on each row
temp = gen_accept_encoded.loc[:10, "CAMEO_INTL_2015"]
print(temp)
print(type(temp))

# To get the first digit
print(int(temp.iloc[0]) // 10)

# TO get the second digit
print(int(temp.iloc[0]) % 10)

In [ ]:
# Create the new features
gen_accept_encoded["WEALTH"] = gen_accept_encoded["CAMEO_INTL_2015"].apply(lambda x: int(x) // 10 if not pd.isna(x) else np.nan)
gen_accept_encoded["LIFE_STAGE"] = gen_accept_encoded["CAMEO_INTL_2015"].apply(lambda x: int(x) % 10 if not pd.isna(x) else np.nan)

In [ ]:
# Sanity Check
gen_accept_encoded.loc[:, ["CAMEO_INTL_2015", "WEALTH", "LIFE_STAGE"]].head(10)

#### Discussion 1.2.2: Engineer Mixed-Type Features

(Double-click this cell and replace this text with your own text, reporting your findings and decisions regarding mixed-value features. Which ones did you keep, which did you drop, and what engineering steps did you perform?)

#### Step 1.2.3: Complete Feature Selection

In order to finish this step up, you need to make sure that your data frame now only has the columns that you want to keep. To summarize, the dataframe should consist of the following:
- All numeric, interval, and ordinal type columns from the original dataset.
- Binary categorical features (all numerically-encoded).
- Engineered features from other multi-level categorical features and mixed features.

Make sure that for any new columns that you have engineered, that you've excluded the original columns from the final dataset. Otherwise, their values will interfere with the analysis later on the project. For example, you should not keep "PRAEGENDE_JUGENDJAHRE", since its values won't be useful for the algorithm: only the values derived from it in the engineered features you created should be retained. As a reminder, your data should only be from **the subset with few or no missing values**.

In [ ]:
gen_accept_encoded.head(5)

In [ ]:
# If there are other re-engineering tasks you need to perform, make sure you
# take care of them here. (Dealing with missing data will come in step 2.1.)



In [ ]:
# Do whatever you need to in order to ensure that the dataframe only contains
# the columns that should be passed to the algorithm functions.
gen_accept_encoded.drop(columns=["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"], inplace=True)

In [ ]:
gen_accept_encoded.head(5)

In [ ]:
gen_accept_encoded.info()

As we can see, we have engineered features and now have a total of 170, and there are no object dtypes.

### Step 1.3: Create a Cleaning Function

Even though you've finished cleaning up the general population demographics data, it's important to look ahead to the future and realize that you'll need to perform the same cleaning steps on the customer demographics data. In this substep, complete the function below to execute the main feature selection, encoding, and re-engineering steps you performed above. Then, when it comes to looking at the customer data in Step 3, you can just run this function on that DataFrame to get the trimmed dataset in a single step.

In [ ]:
def clean_data(df, feature_summary):
    """
    Perform feature trimming, re-encoding, and engineering for demographics
    data
    
    INPUT: Demographics DataFrame
    OUTPUT: Trimmed and cleaned demographics DataFrame
    """
    def get_numeric_value(s):

        try:
            return int(s)
        except ValueError:
            return s

    # Put in code here to execute all main cleaning steps:
    # convert missing value codes into NaNs, ...
    miss_or_unkwn = {}

    for col_label, missing_encoded_str in zip(feature_summary["attribute"], feature_summary["missing_or_unknown"]):

        temp_list = list(missing_encoded_str.replace(" ", "").replace("[","").replace("]","").split(","))

        if len(temp_list) > 0 and temp_list[0] != "":
            miss_or_unkwn[col_label] = [get_numeric_value(s) for s in temp_list]

    for col_label, lst in miss_or_unkwn.items():
        if col_label in df.columns.to_list():
            df[col_label] = df[col_label].replace(lst, np.nan)

    # remove selected columns and rows, ...
    pct_missing_vals = (df.isna().sum() / len(df)) * 100

    drop_cols = list(pct_missing_values[pct_missing_values > 20].index)
    # df.drop(columns=drop_cols, inplace=True)

    data_cols = df.columns.to_list()

    # select, re-encode, and engineer column values.
    cat_features = list(feature_summary[
                        (feature_summary["attribute"].isin(data_cols)) &
                        (feature_summary["type"] == "categorical")]["attribute"])

    cat_features_drop = ["GREEN_AVANTGARDE", "LP_FAMILIE_GROB", "LP_STATUS_GROB", "NATIONALITAET_KZ", "SHOPPER_TYP", "SOHO_KZ", "GEBAEUDETYP"]

    binary_cats = []
    multi_cats = []

    for cat in cat_features:
        if cat in df.columns:

            # Check if binary (two-level) categorical
            if df[cat].nunique() == 2:
                # append to list
                binary_cats.append(cat)
            # check for multi-level categorical, the ones we will not drop
            elif cat not in cat_features_drop:
                multi_cats.append(cat)

    df = df.copy()

    # Silence FutureWarning (very annoying)
    with pd.option_context('future.no_silent_downcasting', True):
        # Then do the replacement with explicit type conversion to avoid FutureWarning

        df["OST_WEST_KZ"] = df["OST_WEST_KZ"].replace(
            {
                'O': 0,
                'W': 1
            }
        ).astype('int64')


    df = pd.get_dummies(df, columns=multi_cats)
    df.drop(columns=list(set(cat_features_drop + drop_cols).intersection(data_cols)), inplace=True)

    map_to_decade = {
        1: 4,
        2: 4,
        3: 5,
        4: 5,
        5: 6,
        6: 6,
        7: 6,
        8: 7,
        9: 7,
        10: 8,
        11: 8,
        12: 8,
        13: 8,
        14: 9,
        15: 9
    }

    # 1 = Mainstream, 0 = Avantgrade
    map_to_movement = {
        1: 1,
        2: 0,
        3: 1,
        4: 0,
        5: 1,
        6: 0,
        7: 0,
        8: 1,
        9: 0,
        10: 1,
        11: 0,
        12: 1,
        13: 0,
        14: 1,
        15: 0
    }

    df["DECADE"] = df.loc[:, "PRAEGENDE_JUGENDJAHRE"].map(map_to_decade)
    df["MOVEMENT"] = df.loc[:, "PRAEGENDE_JUGENDJAHRE"].map(map_to_movement)

    df["WEALTH"] = df["CAMEO_INTL_2015"].apply(lambda x: int(x) // 10 if not pd.isna(x) else np.nan)

    df["LIFE_STAGE"] = df["CAMEO_INTL_2015"].apply(lambda x: int(x) % 10 if not pd.isna(x) else np.nan)

    df.drop(columns=["PRAEGENDE_JUGENDJAHRE", "CAMEO_INTL_2015"], inplace=True)

    # Return the cleaned dataframe.
    return df
    

## Step 2: Feature Transformation

### Step 2.1: Apply Feature Scaling

Before we apply dimensionality reduction techniques to the data, we need to perform feature scaling so that the principal component vectors are not influenced by the natural differences in scale for features. Starting from this part of the project, you'll want to keep an eye on the [API reference page for sklearn](http://scikit-learn.org/stable/modules/classes.html) to help you navigate to all of the classes and functions that you'll need. In this substep, you'll need to check the following:

- sklearn requires that data not have missing values in order for its estimators to work properly. So, before applying the scaler to your data, make sure that you've cleaned the DataFrame of the remaining missing values. This can be as simple as just removing all data points with missing data, or applying an [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) to replace all missing values. You might also try a more complicated procedure where you temporarily remove missing values in order to compute the scaling parameters before re-introducing those missing values and applying imputation. Think about how much missing data you have and what possible effects each approach might have on your analysis, and justify your decision in the discussion section below.
- For the actual scaling function, a [StandardScaler](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) instance is suggested, scaling each feature to mean 0 and standard deviation 1.
- For these classes, you can make use of the `.fit_transform()` method to both fit a procedure to the data as well as apply the transformation to the data at the same time. Don't forget to keep the fit sklearn objects handy, since you'll be applying them to the customer demographics data towards the end of the project.

In [ ]:
# If you've not yet cleaned the dataset of all NaN values, then investigate and
# do that now.
print(f"Sum of NaN values: {gen_accept_encoded.isna().sum().sum()}")
gen_accept_encoded.isnull().sum()

In [ ]:
data_cols = gen_accept_encoded.columns.to_list()

ord_features = list(feature_summary[
                        (feature_summary["attribute"].isin(data_cols)) &
                        (feature_summary["type"] == "ordinal")]["attribute"])

cat_features = list(feature_summary[
                        (feature_summary["attribute"].isin(data_cols)) &
                        (feature_summary["type"] == "categorical")]["attribute"])

mixed_features = list(feature_summary[
                        (feature_summary["attribute"].isin(data_cols)) &
                        (feature_summary["type"] == "mixed")]["attribute"])

numeric_features = list(feature_summary[
                        (feature_summary["attribute"].isin(data_cols)) &
                        (feature_summary["type"] == "numeric")]["attribute"])

# APPEND ENGINEERED MIXED TYPES
mixed_features.extend(["DECADE", "MOVEMENT", "WEALTH", "LIFE_STAGE"])

# EXTEND CATEGORICAL ENCODED FEATURES
to_extend_set = set(data_cols + mixed_features + numeric_features).difference(cat_features)

cat_features.extend(list(to_extend_set))

print(ord_features)
print(cat_features)
print(mixed_features)
print(numeric_features)

print()
print(f"CHECK WE HAVE ALL: {len(set(ord_features + cat_features + mixed_features + numeric_features))}")

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Ordinal transformer
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ('scaler', StandardScaler())
])

# Categorical transformer
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Numeric transformer
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Mixed transformer (treating as categorical)
mixed_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Complete preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ord_features),
        ('cat', categorical_transformer, cat_features),
        ('mix', mixed_transformer, mixed_features),
    ]
)

In [ ]:
# SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy="most_frequent")
imputer = imputer.fit(gen_accept_encoded)
gen_accept_clean = imputer.transform(gen_accept_encoded)
gen_accept_clean = pd.DataFrame(gen_accept_clean, columns=gen_accept_encoded.columns)

In [ ]:
# gen_accept_transformed = preprocessor.fit_transform(gen_accept_encoded)

In [ ]:
# gen_accept_transformed.shape

In [ ]:
# gen_accept_transformed = pd.DataFrame(gen_accept_transformed, columns=gen_accept_encoded.columns)

In [ ]:
# Sanity Check
print(f"Sum of NaN values: {gen_accept_clean.isna().sum().sum()}")
#gen_accept_clean.isnull().sum()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Apply feature scaling to the general population demographics data.
scaler = StandardScaler()
gen_acc_scaled = scaler.fit_transform(gen_accept_clean)
gen_acc_scaled = pd.DataFrame(gen_acc_scaled, columns=gen_accept_clean.columns)

In [ ]:
gen_acc_scaled.head()

### Discussion 2.1: Apply Feature Scaling

(Double-click this cell and replace this text with your own text, reporting your decisions regarding feature scaling.)

### Step 2.2: Perform Dimensionality Reduction

On your scaled data, you are now ready to apply dimensionality reduction techniques.

- Use sklearn's [PCA](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) class to apply principal component analysis on the data, thus finding the vectors of maximal variance in the data. To start, you should not set any parameters (so all components are computed) or set a number of components that is at least half the number of features (so there's enough features to see the general trend in variability).
- Check out the ratio of variance explained by each principal component as well as the cumulative variance explained. Try plotting the cumulative or sequential values using matplotlib's [`plot()`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.plot.html) function. Based on what you find, select a value for the number of transformed features you'll retain for the clustering part of the project.
- Once you've made a choice for the number of components to keep, make sure you re-fit a PCA instance to perform the decided-on transformation.

In [ ]:
# Apply PCA to the data.
n_comps = int(gen_acc_scaled.shape[1] / 2)
print(f"Number of components: {n_comps}")
# pca = PCA(n_components=n_comps)
pca = PCA()
X_pca = pca.fit_transform(gen_acc_scaled)

In [ ]:
# Investigate the variance accounted for by each principal component.
print(pca.explained_variance_ratio_.shape)
pca.explained_variance_ratio_

In [ ]:
n_components = len(pca.explained_variance_ratio_)
ind = np.arange(n_components)
vals = pca.explained_variance_ratio_

plt.figure(figsize=(10, 6))
ax = plt.subplot(111)

c_vals = np.cumsum(vals)

ax.bar(ind, vals)
ax.plot(ind, c_vals)

# ax.xaxis.set_tick_params(width=0)
# ax.yaxis.set_tick_params(width=2, length=12)

ax.set_xlabel("Principal Component")
ax.set_ylabel("Variance-Explained Ratio")
plt.title('Explained Variance Per Principal Component')

In [ ]:
# Determine number of components to reach 90% variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
cumulative_variance

In [ ]:
for n in range(len(cumulative_variance)):
    if cumulative_variance[n] * 100 >= 90:
        print(f"Number of components to reach at least 90% variance: {n}")
        break

In [ ]:
# Re-apply PCA to the data while selecting for number of components to retain.
pca_f = PCA(n_components=97)
X_pca = pca_f.fit_transform(gen_acc_scaled)

In [ ]:
X_pca

In [ ]:
X_pca.shape

### Discussion 2.2: Perform Dimensionality Reduction

(Double-click this cell and replace this text with your own text, reporting your findings and decisions regarding dimensionality reduction. How many principal components / transformed features are you retaining for the next step of the analysis?)

### Step 2.3: Interpret Principal Components

Now that we have our transformed principal components, it's a nice idea to check out the weight of each variable on the first few components to see if they can be interpreted in some fashion.

As a reminder, each principal component is a unit vector that points in the direction of highest variance (after accounting for the variance captured by earlier principal components). The further a weight is from zero, the more the principal component is in the direction of the corresponding feature. If two features have large weights of the same sign (both positive or both negative), then increases in one tend expect to be associated with increases in the other. To contrast, features with different signs can be expected to show a negative correlation: increases in one variable should result in a decrease in the other.

- To investigate the features, you should map each weight to their corresponding feature name, then sort the features according to weight. The most interesting features for each principal component, then, will be those at the beginning and end of the sorted list. Use the data dictionary document to help you understand these most prominent features, their relationships, and what a positive or negative value on the principal component might indicate.
- You should investigate and interpret feature associations from the first three principal components in this substep. To help facilitate this, you should write a function that you can call at any time to print the sorted list of feature weights, for the *i*-th principal component. This might come in handy in the next step of the project, when you interpret the tendencies of the discovered clusters.

In [ ]:
print(pca_f.components_.shape)
pca_f.components_

In [ ]:
# Map weights for the first principal component to corresponding feature names
# and then print the linked values, sorted by weight.
# HINT: Try defining a function here or in a new cell that you can reuse in the
# other cells.
def map_component_weights(pca, df, pc_no):
    feature_names = df.columns
    weights = pca.components_[pc_no]

    assert len(feature_names) == len(weights)

    # weight_mapping = {
    #     n : w
    #     for n, w in zip(feature_names, weights)
    # }

    # weight_mapping = sorted(weight_mapping.items(), key=lambda x: x[1], reverse=True)

    for n, w in sorted(zip(feature_names, weights), key=lambda x: x[1], reverse=True):
        print(f"Feature Name: {n:<25} | Weight: {w:>8.4f}")

In [ ]:
map_component_weights(pca_f, gen_acc_scaled, pc_no=0)

In [ ]:
# Map weights for the second principal component to corresponding feature names
# and then print the linked values, sorted by weight.
map_component_weights(pca_f, gen_acc_scaled, pc_no=1)

In [ ]:
# Map weights for the third principal component to corresponding feature names
# and then print the linked values, sorted by weight.
map_component_weights(pca_f, gen_acc_scaled, pc_no=2)

### Discussion 2.3: Interpret Principal Components

(Double-click this cell and replace this text with your own text, reporting your observations from detailed investigation of the first few principal components generated. Can we interpret positive and negative values from them in a meaningful way?)

## Step 3: Clustering

### Step 3.1: Apply Clustering to General Population

You've assessed and cleaned the demographics data, then scaled and transformed them. Now, it's time to see how the data clusters in the principal components space. In this substep, you will apply k-means clustering to the dataset and use the average within-cluster distances from each point to their assigned cluster's centroid to decide on a number of clusters to keep.

- Use sklearn's [KMeans](http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans) class to perform k-means clustering on the PCA-transformed data.
- Then, compute the average difference from each point to its assigned cluster's center. **Hint**: The KMeans object's `.score()` method might be useful here, but note that in sklearn, scores tend to be defined so that larger is better. Try applying it to a small, toy dataset, or use an internet search to help your understanding.
- Perform the above two steps for a number of different cluster counts. You can then see how the average distance decreases with an increasing number of clusters. However, each additional cluster provides a smaller net benefit. Use this fact to select a final number of clusters in which to group the data. **Warning**: because of the large size of the dataset, it can take a long time for the algorithm to resolve. The more clusters to fit, the longer the algorithm will take. You should test for cluster counts through at least 10 clusters to get the full picture, but you shouldn't need to test for a number of clusters above about 30.
- Once you've selected a final number of clusters to use, re-fit a KMeans instance to perform the clustering operation. Make sure that you also obtain the cluster assignments for the general demographics data, since you'll be using them in the final Step 3.3.

In [ ]:
from sklearn.cluster import KMeans
# Over a number of different cluster counts...
models = [KMeans(n_clusters=k).fit(X_pca) for k in np.arange(start=5, stop=40)]

m_scores = [abs(m.score(X_pca)) for m in models]
m_scores

In [ ]:
wcss = [m.inertia_ for m in models]
wcss

In [ ]:
# Investigate the change in within-cluster distance across number of clusters.
# HINT: Use matplotlib's plot function to visualize this relationship.
plt.plot(np.arange(start=5, stop=40), m_scores, linestyle='--', marker='o', color='b')
plt.xlabel('K')
plt.ylabel('Score')
plt.title('K-Means Clustering Scores');

In [ ]:
# Investigate the change in within-cluster distance across number of clusters.
# HINT: Use matplotlib's plot function to visualize this relationship.
plt.plot(np.arange(start=5, stop=40), wcss, linestyle='--', marker='o', color='b')
plt.xlabel('K')
plt.ylabel('WSCC')
plt.title('WSCC vs K');

In [ ]:
from math import sqrt
# https://jtemporal.com/kmeans-and-elbow-method/#:~:text=Also%2C%20the%20inertia%20isn't%20normalized%2C%20so%20if,tend%20to%20get%20inflated%20in%20multidimensional%20spaces
def optimal_number_of_clusters(wcss):
    x1, y1 = 2, wcss[0]
    x2, y2 = 20, wcss[len(wcss)-1]

    distances = []
    for i in range(len(wcss)):
        x0 = i+2
        y0 = wcss[i]
        numerator = abs((y2-y1)*x0 - (x2-x1)*y0 + x2*y1 - y2*x1)
        denominator = sqrt((y2 - y1)**2 + (x2 - x1)**2)
        distances.append(numerator/denominator)

    return distances.index(max(distances)) + 2

In [ ]:
k = optimal_number_of_clusters(wcss)
k

In [ ]:
# from tqdm.auto import tqdm
from timeit import default_timer as timer

In [ ]:
len(models)

In [ ]:
from sklearn.metrics import silhouette_score

random_state = 42
silhouette_scores = []

start_time = timer()

# Silhouette Score
for k in range(16,35):
    preds = models[k].fit_predict(X_pca)

    silhouette_avg_kmeans = silhouette_score(X_pca, preds, sample_size=10000)
    silhouette_scores.append(silhouette_avg_kmeans)
    print("For k_clusters =", k, "The avg silhouette_score using K-Means is:", silhouette_avg_kmeans)

end_time = timer()
elapsed_time = end_time - start_time
print(f"Calculating Silhouette Scores Complete! Total time: {elapsed_time} seconds")

In [ ]:
preds = models[34].fit_predict(X_pca)

silhouette_avg_kmeans = silhouette_score(X_pca, preds, sample_size=10000)
silhouette_scores.append(silhouette_avg_kmeans)
print("For k_clusters =", k, "The avg silhouette_score using K-Means is:", silhouette_avg_kmeans)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

m_scores_norm = scaler.fit_transform(np.asarray(m_scores).reshape(-1, 1))
silhouette_scores_norm = scaler.fit_transform(np.asarray(silhouette_scores).reshape(-1, 1))

print(m_scores_norm.shape)
print(silhouette_scores_norm.shape)

plt.plot(np.arange(start=16, stop=35), m_scores_norm[16:36], marker='o', label='Inertia (normalized)')
plt.plot(np.arange(start=16, stop=35), silhouette_scores_norm, marker='o', label='Silhouette Score (normalized)')

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Normalized Score (0–1)")
plt.title("Normalized Inertia and Silhouette Score vs k")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Re-fit the k-means model with the selected number of clusters and obtain
# cluster predictions for the general population demographics data.
model_f = KMeans(n_clusters=36).fit(X_pca)
gen_preds = model_f.predict(X_pca)
gen_preds

### Discussion 3.1: Apply Clustering to General Population

(Double-click this cell and replace this text with your own text, reporting your findings and decisions regarding clustering. Into how many clusters have you decided to segment the population?)

### Step 3.2: Apply All Steps to the Customer Data

Now that you have clusters and cluster centers for the general population, it's time to see how the customer data maps on to those clusters. Take care to not confuse this for re-fitting all of the models to the customer data. Instead, you're going to use the fits from the general population to clean, transform, and cluster the customer data. In the last step of the project, you will interpret how the general population fits apply to the customer data.

- Don't forget when loading in the customers data, that it is semicolon (`;`) delimited.
- Apply the same feature wrangling, selection, and engineering steps to the customer demographics using the `clean_data()` function you created earlier. (You can assume that the customer demographics data has similar meaning behind missing data patterns as the general demographics data.)
- Use the sklearn objects from the general demographics data, and apply their transformations to the customers data. That is, you should not be using a `.fit()` or `.fit_transform()` method to re-fit the old objects, nor should you be creating new sklearn objects! Carry the data through the feature scaling, PCA, and clustering steps, obtaining cluster assignments for all of the data in the customer demographics data.

In [ ]:
# Load in the customer demographics data.
customer_data = pd.read_csv("Udacity_CUSTOMERS_Subset.csv", delimiter=';')

In [ ]:
customer_data.isna().sum()

In [ ]:
customer_data.head()
customer_data["OST_WEST_KZ"].fillna(0, inplace=True)
customer_data["OST_WEST_KZ"].isna().sum()

In [ ]:
customers = clean_data(customer_data, feature_summary)

In [ ]:
print(customers.shape)
print(gen_acc_scaled.shape)

In [ ]:
set(gen_acc_scaled.columns.tolist()).difference(set(customers.columns.tolist()))

In [ ]:
customers.head()

In [ ]:
print(f"Customer Clean Data Shape: {customers.shape}")
print(f"General Population Data Shape: {gen_acc_scaled.shape}")

In [ ]:
customers.isna().sum()

In [ ]:
imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
customers_clean = imputer.fit_transform(customers)
customers_clean = pd.DataFrame(customers_clean, columns=customers.columns)

In [ ]:
customers_clean.head()

In [ ]:
assert customers_clean.isna().sum().sum() == 0

In [ ]:
scaler = StandardScaler()
customers_scaled = scaler.fit_transform(customers_clean)
customers_scaled = pd.DataFrame(customers_scaled, columns=customers.columns)
customers_scaled.head()

In [ ]:
pca_cust = pca_f.transform(customers_scaled)
cust_preds = model_f.predict(pca_cust)

### Step 3.3: Compare Customer Data to Demographics Data

At this point, you have clustered data based on demographics of the general population of Germany, and seen how the customer data for a mail-order sales company maps onto those demographic clusters. In this final substep, you will compare the two cluster distributions to see where the strongest customer base for the company is.

Consider the proportion of persons in each cluster for the general population, and the proportions for the customers. If we think the company's customer base to be universal, then the cluster assignment proportions should be fairly similar between the two. If there are only particular segments of the population that are interested in the company's products, then we should see a mismatch from one to the other. If there is a higher proportion of persons in a cluster for the customer data compared to the general population (e.g. 5% of persons are assigned to a cluster for the general population, but 15% of the customer data is closest to that cluster's centroid) then that suggests the people in that cluster to be a target audience for the company. On the other hand, the proportion of the data in a cluster being larger in the general population than the customer data (e.g. only 2% of customers closest to a population centroid that captures 6% of the data) suggests that group of persons to be outside of the target demographics.

Take a look at the following points in this step:

- Compute the proportion of data points in each cluster for the general population and the customer data. Visualizations will be useful here: both for the individual dataset proportions, but also to visualize the ratios in cluster representation between groups. Seaborn's [`countplot()`](https://seaborn.pydata.org/generated/seaborn.countplot.html) or [`barplot()`](https://seaborn.pydata.org/generated/seaborn.barplot.html) function could be handy.
  - Recall the analysis you performed in step 1.1.3 of the project, where you separated out certain data points from the dataset if they had more than a specified threshold of missing values. If you found that this group was qualitatively different from the main bulk of the data, you should treat this as an additional data cluster in this analysis. Make sure that you account for the number of data points in this subset, for both the general population and customer datasets, when making your computations!
- Which cluster or clusters are overrepresented in the customer dataset compared to the general population? Select at least one such cluster and infer what kind of people might be represented by that cluster. Use the principal component interpretations from step 2.3 or look at additional components to help you make this inference. Alternatively, you can use the `.inverse_transform()` method of the PCA and StandardScaler objects to transform centroids back to the original data space and interpret the retrieved values directly.
- Perform a similar investigation for the underrepresented clusters. Which cluster or clusters are underrepresented in the customer dataset compared to the general population, and what kinds of people are typified by these clusters?

In [ ]:
# Compare the proportion of data in each cluster for the customer data to the
# proportion of data in each cluster for the general population.
print(cust_preds)
print(gen_preds)

print()

print(len(np.unique(cust_preds)))
print(len(np.unique(gen_preds)))

print()

print(cust_preds.shape)
print(gen_preds.shape)

print()

print(model_f.labels_)
print(np.unique(model_f.labels_))

# https://stackoverflow.com/questions/28663856/how-do-i-count-the-occurrence-of-a-certain-item-in-an-ndarray
cust_clusters, cust_counts = np.unique(cust_preds, return_counts=True)
gen_clusters, gen_counts = np.unique(gen_preds, return_counts=True)

# Calculate proportions
cust_props = cust_counts / cust_counts.sum()
gen_props = gen_counts / gen_counts.sum()
print()
print(cust_props)
print(gen_props)

In [ ]:
width = 0.35  # width of each bar
clusters = np.unique(model_f.labels_)
x = np.arange(len(clusters))

fig, ax = plt.subplots(figsize=(10, 6))

# Bars
rects1 = ax.bar(x - width/2, gen_props, width, label='General Population')
rects2 = ax.bar(x + width/2, cust_props, width, label='Customers')

ax.set_xlabel("Cluster ID")
ax.set_ylabel("Proportion")
ax.set_title("Customer vs General Population Cluster Distribution")
ax.set_xticks(x)
ax.set_xticklabels(clusters)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Differences in proportions
prop_differences = cust_props - gen_props
prop_diff_df = pd.DataFrame({
    "cluster": clusters,
    "difference": prop_differences
})

prop_diff_df.sort_values(by="difference", ascending=False, inplace=True)
prop_diff_df

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=prop_diff_df, x="cluster", y="difference", palette="coolwarm_r", hue="cluster", legend=False)

plt.axhline(0, color="black", linewidth=1)
plt.title("Customer Over/Underrepresentation by Cluster")
plt.ylabel("gen_props - cust_props")
plt.show()

In [ ]:
# What kinds of people are part of a cluster that is overrepresented in the
# customer data compared to the general population?
centroids = model_f.cluster_centers_
centroids_scaled = pca_f.inverse_transform(centroids)
original_centroids = scaler.inverse_transform(centroids_scaled)

# From the bar graph, we can see that customers in cluster 2 are overrepresented
original_centroids_df = pd.DataFrame(original_centroids, columns=subset_accept_encoded.columns)
original_centroids_df.head(9)

In [ ]:
print(original_centroids_df.shape)
original_centroids_df.loc[2]

In [ ]:
centroids_scaled_df = pd.DataFrame(centroids_scaled, columns=subset_accept_encoded.columns)
cluster_2_scores = centroids_scaled_df.loc[2]
important_features = cluster_2_scores.abs().sort_values(ascending=False)

top_over = important_features.head(10)
vals = cluster_2_scores[top_over.index]

plt.figure(figsize=(10,6))
plt.barh(top_over.index, vals,
         color=["tab:red" if v>0 else "tab:blue" for v in vals])
plt.title("Cluster 2 – Top Features (z-scores)")
plt.xlabel("Z-score relative to overall mean")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
overrep_list = top_over.index.to_list()
original_centroids_df.loc[2, overrep_list]

In [ ]:
# What kinds of people are part of a cluster that is underrepresented in the
# customer data compared to the general population?
cluster_4_scores = centroids_scaled_df.loc[4]
important_features_2 = cluster_4_scores.abs().sort_values(ascending=False)
important_features_2.head(10)

top_under = important_features_2.head(10)
vals = cluster_4_scores[top_under.index]

plt.figure(figsize=(10,6))
plt.barh(top_under.index, vals,
         color=["tab:red" if v>0 else "tab:blue" for v in vals])
plt.title("Cluster 4 – Top Features (z-scores)")
plt.xlabel("Z-score relative to overall mean")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
underrep_list = top_under.index.to_list()
original_centroids_df.loc[4, underrep_list]

### Discussion 3.3: Compare Customer Data to Demographics Data

(Double-click this cell and replace this text with your own text, reporting findings and conclusions from the clustering analysis. Can we describe segments of the population that are relatively popular with the mail-order company, or relatively unpopular with the company?)

> Congratulations on making it this far in the project! Before you finish, make sure to check through the entire notebook from top to bottom to make sure that your analysis follows a logical flow and all of your findings are documented in **Discussion** cells. Once you've checked over all of your work, you should export the notebook as an HTML document to submit for evaluation. You can do this from the menu, navigating to **File -> Download as -> HTML (.html)**. You will submit both that document and this notebook for your project submission.